In [1]:
import pandas as pd
import re

In [2]:
artistless = pd.read_csv("data/features-annotated-artistless-with-lyrics-and-languages.csv")
featuring = artistless[artistless["Featuring"] == 1]

In [3]:
featuring

,id,Rank,Title,Artists,Featuring,Date,Danceability,Energy,Loudness,Speechiness,...,Instrumentalness,Valence,Nationality,Continent,Points (Total),artist,lyrics,detected_languages,num_languages,language_confidences
8,64BuLY7ggczC17w5LSUlSE,1,creepin' (with the weeknd & 21 savage),"21 savage, metro boomin, the weeknd",1,06/12/2022,0.713,0.611,-5993.0,0.051,...,0.000,0.173,united states,anglo-america,200,21 savage,NaN,NaN,0,NaN
18,27NovPIUIRrOZoCHxABJwK,2,industry baby (feat. jack harlow),"jack harlow, lil nas x",1,16/09/2021,0.736,0.704,-7409.0,0.062,...,0.000,0.894,united states,anglo-america,199,jack harlow,"(\nD-D-Daytrip took it to ten, hey\n)\nBaby ba...",ENGLISH,1,ENGLISH: 1.0
26,2Imq77E6CEJR5TPUxlJvTY,3,all the stars (with sza),kendrick lamar,1,28/02/2018,0.532,0.642,-4891.0,0.111,...,0.000,0.536,united states,anglo-america,198,kendrick lamar,NaN,NaN,0,NaN
27,6ujbuB0L7JKxmpQzAoCMuK,3,die for you (with ariana grande) - remix,"ariana grande, the weeknd",1,15/03/2023,0.570,0.486,-7668.0,0.075,...,0.000,0.528,canada,anglo-america,198,ariana grande,NaN,NaN,0,NaN
31,24ySl2hOPGCDcxBxFIqWBu,3,rain on me (with ariana grande),lady gaga,1,28/05/2020,0.672,0.855,-3764.0,0.040,...,0.000,0.646,united states,anglo-america,198,lady gaga,I didn't ask for a free ride\nI only asked you...,ENGLISH,1,ENGLISH: 1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9145,70zRDnEkOhVb51fZA13khd,200,yallah goodbye (feat. gringo),summer cem,1,06/07/2019,0.875,0.738,-6172.0,0.175,...,0.023,0.210,germany,europe,1,summer cem,"Miksu\n\nRaindrop, drop top\nSenin anneni Glad...",TURKISH,1,TURKISH: 1.0
9148,0Odu228D0yGcFySGuxU4rY,200,the take (feat. chris brown),tory lanez,1,27/04/2020,0.464,0.547,-8137.0,0.339,...,0.000,0.423,canada,anglo-america,1,tory lanez,Take you down\nWhat I'm about\nTake you down\n...,ENGLISH,1,ENGLISH: 1.0
9149,0PvFJmanyNQMseIFrU708S,200,for the night (feat. lil baby & dababy),pop smoke,1,25/08/2021,0.823,0.586,-6606.0,0.200,...,0.000,0.347,united states,anglo-america,1,pop smoke,"CashMoneyAP\nÖldür onları, al onları, al onlar...",TURKISH,1,TURKISH: 1.0
9157,7zQirOExB0VR8yWUOqYeio,200,dolla sign slime (feat. megan thee stallion),lil nas x,1,27/09/2021,0.923,0.533,-6016.0,0.262,...,0.000,0.587,united states,anglo-america,1,lil nas x,Day-Day-Daytrip took it to ten (\nHey\n)\n\nY'...,ENGLISH,1,ENGLISH: 1.0


In [4]:
# Take a string with featuring terms and produce a comma separated list of featured artists
def strip_feature(artist_str):
    # Check for "with <artists1> feat/ft. <artists2>" and "with <artists1> & feat/ft. <artists2>"
    # Handle "with <artists1> & feat/ft. <artists2>" and "with <artists1> feat/ft. <artists2>"
    s = artist_str
    def join_groups(*groups):
        return ','.join(g.strip() for g in groups if g)

    s = re.sub(
        r'with\s+([^\(\)\[\]]+?)\s*&\s*(?:feat|ft)\.?\s*([^\(\)\[\]]+)',
        lambda m: join_groups(m.group(1), m.group(2)),
        s,
        flags=re.IGNORECASE
    )
    s = re.sub(
        r'with\s+([^\(\)\[\]]+?)\s*(?:feat|ft)\.?\s*([^\(\)\[\]]+)',
        lambda m: join_groups(m.group(1), m.group(2)),
        s,
        flags=re.IGNORECASE
    )
    # Handle "<artists1> & <artists2> feat/ft. <artists3>"
    s = re.sub(
        r'([^\(\)\[\]]+?)\s*&\s*([^\(\)\[\]]+?)\s*(?:feat|ft)\.?\s*([^\(\)\[\]]+)',
        lambda m: join_groups(m.group(1), m.group(2), m.group(3)),
        s,
        flags=re.IGNORECASE
    )
    # Handle "<artists1> feat/ft. <artists2>"
    s = re.sub(
        r'([^\(\)\[\]]+?)\s*(?:feat|ft)\.?\s*([^\(\)\[\]]+)',
        lambda m: join_groups(m.group(1), m.group(2)),
        s,
        flags=re.IGNORECASE
    )

    # Remove "duet with <artist>"
    s = re.sub(r'duet with\s*[^\(\)\[\],]+', '', s, flags=re.IGNORECASE)

    # Remove all featuring terms in data-set 
    s = re.sub(r'\b(feat\.?\s*|featuring\s*|starring\s*|with\s*|con\s*|ft\.?\s*| remix edit\s*|w/\s*)\b\.?', '', s, flags=re.IGNORECASE)
    # Replace splitters with comma
    s = re.sub(r'\s*&\s*', ',', s) # <artist1> & <artist2>
    s = re.sub(r'\s* and \s*', ',', s) # <artist 1> and <artist2>
    s = re.sub(r'(?<!\,)\s+x\s+', ',', s) # <artist 1> x <artist2>, but not ", x ambassadors"
    s = re.sub(r'\s*, \s*', ',', s) # <artist1>, <artist2>
    s = re.sub(r'\s*/\s*', ',', s) # <artist1> / <artist2>
    s = re.sub(r'\s*vs\.\s*', ',', s, flags=re.IGNORECASE) # <artist1> vs. <artist2>
    return s.strip()

# Remove additional information about artists, e.g. "of/from <band>"
# Many of these are for individual cases
def parse_artist(artist):
    artist = re.sub(r'(of|from)\s+[^,]+', '', artist, flags=re.IGNORECASE) # <artist> of/from <band>
    artist = re.sub(r'his orchestra', '', artist, flags=re.IGNORECASE) # <artist> and his orchestra
    artist = re.sub(r'\'s orchestra', '', artist, flags=re.IGNORECASE) # <artist>'s orchestra
    artist = re.sub(r'chorus', '', artist, flags=re.IGNORECASE) # <artist> and chorus
    artist = re.sub(r'present\s+[^,]+', '', artist, flags=re.IGNORECASE) # <artists> present <show>
    artist = re.sub(r'as the people', '', artist, flags=re.IGNORECASE) # <artist> as the people
    artist = re.sub(r'remix;|remix', '', artist, flags=re.IGNORECASE) # <artist> remix or remix;
    artist = re.sub(r' radio edit', '', artist, flags=re.IGNORECASE) # <artist> radio edit
    artist = re.sub(r'rration by ', '', artist, flags=re.IGNORECASE) # rration by <artist>, note that "na" is absent from the dataset due to terrible cleaning
    artist = re.sub(r'\.*$', '', artist.strip())
    return artist

featured_artists = set()

featured_artists_id = {}
for _, row in featuring.iterrows():
    song_id = row.iloc[0]
    # Remove "." from the end of titles (helps with some featuring)
    title = row.iloc[2].rstrip(".")

    # Check for bracketed strings, e.g. (feat. sza) or [with future & chris brown]
    if (re.search(r"\(.*\)", title) or re.search(r"\[.*\]", title)):
        # Songs may have multiple bracket strings, e.g. superhero (heroes & villains) [with future & chris brown]
        brackets = re.findall(r"(\(.*?\)|\[.*?\])", title)

        # For each found bracket we parse to find artists
        for bracket in brackets:
            bracket = bracket.strip("()[]")
            stripped = strip_feature(bracket)

            # If we've changed something then we've successfully matched something in the regular expression and will have an artist list
            if stripped != bracket:
                featured = list(filter(lambda x: x != "", map(parse_artist, stripped.split(","))))
    
    # Non-bracketed features almost always come after a -
    elif len(split := title.split(" - ")) >= 2:
            stripped = strip_feature(split[-1])
            featured = list(filter(lambda x: x != "", map(parse_artist, stripped.split(","))))

    # Those which do not come after either feat. or ft.
    elif " feat. " in title:
        featured = title.split("feat. ")[1].split(", ")
    elif " ft. " in title:
        featured = title.split("ft. ")[1].split(", ")

    # Extend the featured artists set with the newly found featured artists
    featured_artists = featured_artists | set(featured)
    featured_artists_id[song_id] = ", ".join(sorted(featured))
    

In [5]:
# Split the 'Artists' column by comma and flatten the list
all_artists = set(artistless['Artists'].str.split(',').explode().str.strip())
for artist in (featured_artists - all_artists):
    print(artist)

judah
pr1svx edit
orietta berti
lagola
phresher
9lokknine
lenny tavarez
drefgold
mitchell ayres
bootsy collins
anthony hamilton
the puppini sisters
empress of
nyla
lagique
tierra whack
norma jean martine
bono
kellin quinn
fabolous
kr ver
jake gosling
wale
daryl's version
swarmz
kranium
sly pyper
nomcebo zikode
lenny santos
s-x
dram
stun gambino
bizzy banks
mavis staples
adam levine
pharrell
saudi
chief keef
jack white
kiddo
erroll garner
mez
ray blk
lauren alai
sammy
john martin
babes wodumo
guilherme
princess nokia
fuse odg
morray
peven everett
guapdad 4000
colbie caillat
dan caplen
ryan riback
james gillespie
joe janiak
jamie principle
geolier
mufasa
john lindahl
ybn cordae
emilee
gza
aj tracey
maty noyes
sea shanty
maxo kream
alida
j. balvin
towkio
mae muller
ilovemakonnen
gary lightbody
ryan tedder
xavier idoo
lennox
dj yuri martins
bülow
dom kennedy
trettmann
a-trak
jackpot scotty wotty
currensy
jt
descemer bueno
ape drums
masta killa
buni
childish major
travi
seezyn
gringo
kill e

In [6]:
# Remove boolean value and insert featuring artists, reindexing the columns appropriately
artistless.drop(columns=["Featuring"], inplace=True)
artistless = artistless.join(
    pd.DataFrame.from_dict(featured_artists_id, orient='index', columns=["Featuring Artists"]),
    on="id"
).reindex(
    columns=list(artistless.columns[:artistless.columns.get_loc("Artists")+1]) +
            ["Featuring Artists"] +
            list(artistless.columns[artistless.columns.get_loc("Artists")+1:])
)
artistless.rename(columns={"Artists": "Primary Artists"}, inplace=True)

# Remove featuring artists from primary artists
artistless["Primary Artists"] = artistless.apply(
    lambda row: ", ".join(sorted(set(row["Primary Artists"].split(", ")) - set(str(row["Featuring Artists"]).split(", ")) if pd.notnull(row["Featuring Artists"]) else set(row["Primary Artists"].split(", ")))),
    axis=1
)

# Remove "artist" column
artistless.drop(columns=["artist"], inplace=True)